# Feature Engineering

This notebook is designed to prepare our data for machine learning models. Since I am not scaling any data nor creating our train/test split, this will be a separate notebook from preprocessing. The transformations here include column removal, deduplication to first patient encounter, missing value handling, ICD-9 diagnosis categorization, medication consolidation, discharge filtering, and categorical encoding.

In [35]:
import pandas as pd
import numpy as np
import sqlite3

conn = sqlite3.connect('../diabetes-readmission.db')

df = pd.read_sql("""
    SELECT * FROM encounters
""", conn)

conn.close()

df_clone = df
df_clone.head()

,encounter_id,patient_nbr,race,gender,age,weight,admission_type_id,discharge_disposition_id,admission_source_id,time_in_hospital,...,citoglipton,insulin,glyburide-metformin,glipizide-metformin,glimepiride-pioglitazone,metformin-rosiglitazone,metformin-pioglitazone,change,diabetesMed,readmitted
0,2278392,8222157,Caucasian,Female,[0-10),?,6,25,1,1,...,No,No,No,No,No,No,No,No,No,NO
1,149190,55629189,Caucasian,Female,[10-20),?,1,1,7,3,...,No,Up,No,No,No,No,No,Ch,Yes,>30
2,64410,86047875,AfricanAmerican,Female,[20-30),?,1,1,7,2,...,No,No,No,No,No,No,No,No,Yes,NO
3,500364,82442376,Caucasian,Male,[30-40),?,1,1,7,2,...,No,Up,No,No,No,No,No,Ch,Yes,NO
4,16680,42519267,Caucasian,Male,[40-50),?,1,1,7,1,...,No,Steady,No,No,No,No,No,Ch,Yes,NO


## Dropped Columns

Columns that showed high rates of missing or unreliable data, or low significance to the model, were dropped. Weight (96% missing), medical specialty (49% missing), and max glucose serum (94% missing, no contextual metadata) were removed as imputation would have produced unreliable values. Payer code was excluded to focus the model on factors within hospital control rather than socioeconomic proxies. Encounter ID was removed as a non-predictive unique identifier.

In [36]:
columns_to_drop = ['weight', 'medical_specialty', 'payer_code', 'max_glu_serum', 'encounter_id', 'change', 'diabetesMed']

df_clone.drop(columns=columns_to_drop, inplace=True)
df_clone.shape

(101766, 43)

## Deduplication

Since the goal of my project is to identify why a diabetic patient might be readmitted to the hospital, those  who appeared multiple times across the dataset were deduplicated to the first encounter. This reduced the possibility of data leakage by not allowing the same patient to be in both training and test sets

In [37]:
df_clone.drop_duplicates(subset='patient_nbr', keep='first', inplace=True)
df_clone.drop(columns='patient_nbr', inplace=True)
df_clone = df_clone.reset_index(drop=True)
df_clone.shape

(71518, 42)

## Medication Consolidation

The 23 individual diabetes medication columns were consolidated into a single binary indicator, any_med_change, representing whether any diabetes medication dosage was adjusted during the admission. This differs from the original change column, which tracked changes across all medications without specifying which ones. The consolidation reflects the EDA finding that medication changes, rather than which specific medications were prescribed, showed the stronger signal for readmission risk.

In [38]:
all_med_cols = ['metformin', 'repaglinide', 'nateglinide', 'chlorpropamide',
                'glimepiride', 'acetohexamide', 'glipizide', 'glyburide',
                'tolbutamide', 'pioglitazone', 'rosiglitazone', 'acarbose',
                'miglitol', 'troglitazone', 'tolazamide', 'examide',
                'citoglipton', 'insulin', 'glyburide-metformin',
                'glipizide-metformin', 'glimepiride-pioglitazone',
                'metformin-rosiglitazone', 'metformin-pioglitazone']

# Create binary change indicator before dropping
df_clone['any_med_change'] = (df_clone[all_med_cols].isin(['Up', 'Down'])).any(axis=1).astype(int)

# Drop all individual medication columns
df_clone.drop(columns=all_med_cols, inplace=True)
df_clone.shape

(71518, 20)

## Missing Data Handling

The remaining columns with missing data were categorical. Rather than dropping these rows or imputing values, a separate 'Unknown' category was created for each. The Unknown categories themselves carry signal; for A1C specifically, the absence of a test likely reflects admission severity rather than random missingness, as doctors tend not to order A1C when a patient is admitted for a non-diabetic crisis. Preserving this distinction allows the model to learn from the pattern of missingness itself rather than treating it as noise.

In [39]:
def replace_missing_data(entry):
    entry = str(entry).strip()
    if entry == "?":
        return "Unknown"
    return entry

def categorize_icd9(code):
    code = str(code).strip()
    
    if code == '?':
        return "Unknown"
    
    if code.startswith('V'):
        return "Supplementary"
    if code.startswith("E"):
        return "External"
    
    try:
        num = float(code)
    except ValueError:
        return "Other"
    
    if num < 140:
        return "Infectious Disease"
    elif num >=140 and num < 240:
        return "Neoplasms"
    elif num >=240 and num < 280:
        if num >= 250 and num < 251:
            return "Diabetes"
        return "Endocrine"
    elif num >=280 and num < 290:
        return "Blood Disease"
    elif num >=290 and num < 320:
        return "Behavioral"
    elif num >=320 and num < 390:
        return "Nervous System"
    elif num >=390 and num < 460:
        return "Circulatory"
    elif num >=460 and num < 520:
        return "Respiratory"
    elif num >=520 and num < 580:
        return "Digestive"
    elif num >=580 and num < 630:
        return "Genitourinary"
    elif num >=630 and num < 680:
        return "Pregnancy"
    elif num >=680 and num < 710:
        return "Skin"
    elif num >=710 and num < 740:
        return "Musculoskeletal"
    elif num >=740 and num < 760:
        return "Congenital Anomalies"
    elif num >=760 and num < 780:
        return "Perinatal"
    elif num >=780 and num < 800:
        return "Ill-defined Conditions"
    elif num >=800 and num < 1000:
        return "Injury and Poisoning"
    
    

    return "Other"


In [40]:
df_clone['race'] = df['race'].apply(replace_missing_data)
df_clone['diag_1'] = df['diag_1'].apply(categorize_icd9)
df_clone['diag_2'] = df['diag_2'].apply(categorize_icd9)
df_clone['diag_3'] = df['diag_3'].apply(categorize_icd9)
df_clone['A1Cresult'] = df['A1Cresult'].fillna('Unknown')

print(df_clone['diag_1'].value_counts())
print(df_clone['race'].value_counts(normalize=True))
df_clone.shape

diag_1
Circulatory               16449
Respiratory                4769
Digestive                  4671
Diabetes                   4457
Ill-defined Conditions     4172
Injury and Poisoning       3448
Musculoskeletal            2955
Genitourinary              2370
Neoplasms                  1998
Endocrine                  1424
Skin                       1277
Infectious Disease         1232
Behavioral                 1186
Supplementary               760
Nervous System              600
Blood Disease               461
Pregnancy                   406
Congenital Anomalies         31
Unknown                       9
Name: count, dtype: int64
race
Caucasian          0.738548
AfricanAmerican    0.200513
Unknown            0.023028
Hispanic           0.018643
Other              0.013802
Asian              0.005467
Name: proportion, dtype: float64


(71518, 20)

## Admissions Data

Since the method of patient admission could be a strong signal for readmission risk, admission type and admission source were retained and mapped to meaningful labels using the provided IDs mapping file before encoding. Encoding raw numeric IDs directly would imply a false ordinal relationship, as ID 3 is not inherently greater than ID 1, whereas label-based encoding treats each admission type as a distinct category. Rare and clinically ambiguous categories were consolidated into an 'Other' bucket to reduce dimensionality without losing the signal from the meaningful categories.

In [41]:
admission_type_map = {
    1:'Emergency',
    2:'Urgent',
    3:'Elective',
    4:'Newborn',
    5:'Not Available',
    6:'NULL',
    7:'Trauma Center',
    8:'Not Mapped'
}

admission_type_other = ['NULL', 'Not Available', 'Not Mapped', 'Trauma Center', 'Newborn']

df_clone['admission_type'] = df_clone['admission_type_id'].map(admission_type_map)
df_clone['admission_type'] = df_clone['admission_type'].replace(admission_type_other, 'Other')
df_clone.drop(columns='admission_type_id', inplace=True)
print(df_clone['admission_type'].value_counts())

admission_type
Emergency    36490
Elective     13917
Urgent       13028
Other         8083
Name: count, dtype: int64


In [42]:
admission_source_map = {
    1:'Physician Referral',
    2:'Clinical Referral',
    3:'HMO Referral',
    4:'Transfer from a hospital',
    5:'Transfer from a Skilled Nursing Facility (SNF)',
    6:'Transfer from another health care facility',
    7:'Emergency Room',
    8:'Court/Law Enforcement',
    9:'Not Available',
    10:'Transfer from critial access hospital',
    11:'Normal Delivery',
    12:'Premature Delivery',
    13:'Sick Baby',
    14:'Extramural Birth',
    15:'Not Available',
    17:'NULL',
    18:'Transfer From Another Home Health Agency',
    19:'Readmission to Same Home Health Agency',
    20:'Not Mapped',
    21:'Unknown/Invalid',
    22:'Transfer from hospital inpt/same fac reslt in a sep claim',
    23:'Born inside this hospital',
    24:'Born outside this hospital',
    25:'Transfer from Ambulatory Surgery Center',
    26:"Transfer from Hospice"
}

admission_source_other = [
    'NULL', 
    'Not Mapped', 
    'HMO Referral', 
    'Not Available', 
    'Court/Law Enforcement',
    'Transfer from critial access hospital',
    'Transfer from hospital inpt/same fac reslt in a sep claim',
    'Extramural Birth',
    'Transfer from Ambulatory Surgery Center',
    'Normal Delivery',
    'Sick Baby'
]

df_clone['admission_source'] = df_clone['admission_source_id'].map(admission_source_map)
df_clone['admission_source'] = df_clone['admission_source'].replace(admission_source_other, "Other")
df_clone.drop(columns='admission_source_id', inplace=True)
print(df_clone['admission_source'].value_counts())

admission_source
Emergency Room                                    38290
Physician Referral                                22007
Other                                              5366
Transfer from a hospital                           2583
Transfer from another health care facility         1801
Clinical Referral                                   926
Transfer from a Skilled Nursing Facility (SNF)      545
Name: count, dtype: int64


In [43]:
df_clone = pd.get_dummies(df_clone, columns=['age', 'race', 'gender', 'diag_1', 'diag_2', 'diag_3', 'A1Cresult', 'admission_type', 'admission_source'], drop_first=True)
df_clone.shape
df_clone.head()

,discharge_disposition_id,time_in_hospital,num_lab_procedures,num_procedures,num_medications,number_outpatient,number_emergency,number_inpatient,number_diagnoses,readmitted,...,A1Cresult_Unknown,admission_type_Emergency,admission_type_Other,admission_type_Urgent,admission_source_Emergency Room,admission_source_Other,admission_source_Physician Referral,admission_source_Transfer from a Skilled Nursing Facility (SNF),admission_source_Transfer from a hospital,admission_source_Transfer from another health care facility
0,25,1,41,0,1,0,0,0,1,NO,...,True,False,True,False,False,False,True,False,False,False
1,1,3,59,0,18,0,0,0,9,>30,...,True,True,False,False,True,False,False,False,False,False
2,1,2,11,5,13,2,0,1,6,NO,...,True,True,False,False,True,False,False,False,False,False
3,1,2,44,1,16,0,0,0,7,NO,...,True,True,False,False,True,False,False,False,False,False
4,1,1,51,0,8,0,0,0,5,NO,...,True,True,False,False,True,False,False,False,False,False


## Discharge Filtering

Since it isn't uncommon for patients to pass away in a hospital or be discharge to hospice (where they most likely will not be readmitted), I removed all of the patients who had been discharged under these codes. This removes the possibility of the model learning that the most effective way to keep people from being readmitted is for them to die.

In [44]:
codes = [11, 13, 14, 19, 20, 21]
df_clone = df_clone[~df_clone['discharge_disposition_id'].isin(codes)]
df_clone.drop(columns=['discharge_disposition_id'], inplace=True)
df_clone['readmitted'] = (df_clone['readmitted'] != 'NO').astype(int)

df_clone.head()

,time_in_hospital,num_lab_procedures,num_procedures,num_medications,number_outpatient,number_emergency,number_inpatient,number_diagnoses,readmitted,any_med_change,...,A1Cresult_Unknown,admission_type_Emergency,admission_type_Other,admission_type_Urgent,admission_source_Emergency Room,admission_source_Other,admission_source_Physician Referral,admission_source_Transfer from a Skilled Nursing Facility (SNF),admission_source_Transfer from a hospital,admission_source_Transfer from another health care facility
0,1,41,0,1,0,0,0,1,0,0,...,True,False,True,False,False,False,True,False,False,False
1,3,59,0,18,0,0,0,9,1,1,...,True,True,False,False,True,False,False,False,False,False
2,2,11,5,13,2,0,1,6,0,0,...,True,True,False,False,True,False,False,False,False,False
3,2,44,1,16,0,0,0,7,0,1,...,True,True,False,False,True,False,False,False,False,False
4,1,51,0,8,0,0,0,5,0,0,...,True,True,False,False,True,False,False,False,False,False


## Log Transformation

Prior utilization features (number_inpatient, number_emergency, number_outpatient) showed zero-inflated distributions with extreme outliers: the IQR for all three was 0, making standard scaling ineffective. A log transformation (log1p) was applied before scaling to compress extreme values while preserving relative differences between patients.

In [45]:
for col in ['number_inpatient', 'number_emergency', 'number_outpatient']:
    df[col] = np.log1p(df[col])

In [46]:
conn = sqlite3.connect('../diabetes-readmission.db')
df_clone.to_sql('encounters_clean', conn, if_exists='replace', index=False)
conn.close()

print("encounters_clean saved successfully")
print(f"Shape: {df_clone.shape}")

encounters_clean saved successfully
Shape: (69973, 94)


## Next Steps

Now that the data can be read by a machine learning model, the next step is to standardize the continuous features and split the dataset into training and test sets. Given the outlier-heavy nature of the dataset observed during EDA, RobustScaler will be applied to the continuous numeric features rather than StandardScaler, as it uses median and IQR rather than mean and standard deviation, making it resistant to the influence of extreme values. One-hot encoded binary columns will not be scaled, as they already exist on a 0-1 scale.